In [1]:
import requests as req
from bs4 import BeautifulSoup
import time
import os
from dotenv import load_dotenv
from PIL import Image
from io import BytesIO
import re
import google.generativeai as genai

c:\Users\USER\miniconda3\envs\dsde\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\USER\AppData\Local\Temp\ipykernel_3156\859815715.py:9: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [2]:
load_dotenv()

True

In [3]:
m_url = "https://www.mycourseville.com/?q=onlinecourse/quiz/"
urls_id = [
    1286311,
    1251971,
    1284559,
    1284754,
    1284805,
    1284852,
]

In [4]:
htmls = []
my_cookie = os.environ["TOKEN_COOKIE"]

In [5]:
header = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36",
    "Referer": "https://www.mycourseville.com/?q=onlinecourse/course/51604",
    "sec-ch-ua": '"Chromium";v="146", "Not-A.Brand";v="24", "Google Chrome";v="146"',
    "sec-ch-ua-mobile": "?0",
    "sec-ch-ua-platform": '"Windows"',
    "Upgrade-Insecure-Requests": "1",
    "Cookie": my_cookie
}

In [6]:
for id in urls_id:
    response = req.post(url=f"{m_url}{id}", headers=header)
    time.sleep(1)
    if response.status_code == 200:
        html = response.text
        htmls.append(html)  
    else:
        print(f"id: {id} course error") 

In [7]:
question_each_course = []
answer_each_course = []
question_image_eachcourse = []
questions_code = []
titles = []

In [8]:
for html in htmls:
    soup = BeautifulSoup(html, "lxml")
    title = soup.select_one("a[href*='?q=onlinecourse/course/'][title='Go to the learning path of the course']").text
    titles.append(title)
    questions = soup.select("div.cvocp-quiz-item")
    question_course = []
    question_image_course = []
    question_code = []
    answer_course = []
    question_name_image = []
    for i, question in enumerate(questions):
        questions_part = question.select_one("div[data-part='question-body']")
        responses_part = question.select("div[data-part='choice-item']")
        have_imgs_question = questions_part.select("p")
        have_code_question = questions_part.select("pre")
        imgs_question = []
        code_question = []
        q_parts = [] 
        q = ""
        if len(have_imgs_question) != 0:
            for p in have_imgs_question:
                img = p.select_one("img")
                if img:
                    url = img.get("src")
                    if not url.startswith("http"):
                        url = "https://www.mycourseville.com/" + url
                    res = req.get(url) 
                    if res.status_code == 200:
                        # print(i)
                        # print(url)
                        image = Image.open(BytesIO(res.content))
                        imgs_question.append(image)
                    else:
                        print(f"โหลดรูปไม่สำเร็จ: {res.status_code}")
                else:
                    text = p.get_text(strip=True)

                    if text:
                        if len(text) > 3: 
                            q_parts.append(text)

            q = "\n".join(q_parts)
        else:
            q = questions_part.text.strip()
        if len(have_code_question) != 0:
            for c in have_code_question:
                code_question.append(c.text.strip())

        question_course.append(q)
        question_image_course.append(imgs_question)
        question_code.append(code_question)

        choics = []
        for choice in responses_part:
            ans = choice.select_one("span[data-part='choice-content']")
            choics.append(ans.text.strip())
        
        answer_course.append(choics)
    # print(question_image_course)
    # print(question_course)
    question_each_course.append(question_course)
    answer_each_course.append(answer_course)
    question_image_eachcourse.append(question_image_course)
    questions_code.append(question_code)

In [11]:
question_each_course[0]

['ตัวเลือกใดแสดงแนวคิดของการทำงานโดยคอมพิวเตอร์เชิงไฟฟ้า',
 'ทรานซิสเตอร์ถูกใช้อย่างไรในระบบคอมพิวเตอร์',
 'กำหนดข้อมูลเลขฐานสอง ขนาด 8 หลักเป็น 10010000 ตัวเลือกใดถูกต้อง',
 'หากกำหนด A = 0 และ B = 1 ตัวเลือกใดเป็นผลลัพธ์ของวงจรต่อไปนี้',
 'หากกำหนด A = 0100 และ B = 0010 ตัวเลือกใดเป็นผลลัพธ์ของ A AND B',
 'จากตัวเลือกต่อไปนี้ หน่วยความจำแบบใดทำงานได้เร็วที่สุด',
 'ตัวเลือกใดเป็นส่วนคำนวณของคอมพิวเตอร์',
 'คอมพิวเตอร์ประกอบด้วยฮาร์ดแวร์ชิ้นต่างๆ จำนวนมาก การทำงานร่วมกันของอุปกรณ์ต่างๆ เหล่านี้เกิดขึ้นได้อย่างไร',
 'ภาษาแอสเซมบลีคืออะไร',
 'การมี core หรือแกนประมวลผล มากขึ้น มีประโยชน์อย่างไร',
 'ตัวเลือกใดเป็นหน้าที่ของระบบปฏิบัติการ',
 'เหตุใดการสร้างไฟล์ เช่น ไฟล์ภาพ JPG บนโทรศัพท์เคลื่อนที่ สามารถนำไปเปิดในเครื่องอื่นๆ เช่น คอมพิวเตอร์ตั้งโต๊ะ ได้เช่นกัน แม้เครื่องนั้นจะมีสถาปัตยกรรมที่ต่างกัน',
 'ตัวเลือกใดถูกต้องเกี่ยวกับการเก็บข้อมูลภาพ',
 'หากข้อมูลทุกรูปแบบในคอมพิวเตอร์ปัจจุบันเก็บโดยใช้เลขฐานสองทั้งหมด ตัวเลือกใดต่อไปนี้ถูกต้อง',
 'หากกำหนด A = 0 และ B = 1 ตัวเลือกใดเป็นผลลัพ

In [9]:
def clean_filename(name):
    return re.sub(r'[\\/*?:"<>|]', "", name).strip()

In [ ]:
genai.configure(api_key=os.environ.get("GEMINI_API_KEY"))
model = genai.GenerativeModel('gemini-3.1-flash-lite-preview') # <- choose model with your self 
print("เริ่มส่งข้อมูลให้ Gemini วิเคราะห์...")
for c_idx, title in enumerate(titles):
    safe_title = clean_filename(title)
    file_path = f"{safe_title}.md"
    
    print(f"\nกำลังประมวลผลคอร์ส: {title}...")
    
    with open(file_path, "w", encoding="utf-8") as md_file:
        md_file.write(f"# คอร์ส: {title}\n\n")
        
        questions = question_each_course[c_idx]
        images = question_image_eachcourse[c_idx]
        codes = questions_code[c_idx]
        answers = answer_each_course[c_idx]
        
        # 3. วนลูปตามจำนวนข้อในคอร์สนี้
        for q_idx in range(len(questions)):
            print(f"  - กำลังตอบข้อที่ {q_idx + 1}/{len(questions)}")
            
            q_text = questions[q_idx]
            q_imgs = images[q_idx]
            q_code = codes[q_idx]
            q_choices = answers[q_idx]
            
            prompt = f"จงตอบคำถามต่อไปนี้ และอธิบายเหตุผลอย่างละเอียด:\n\n"
            prompt += f"คำถาม: {q_text}\n\n"
            
            if q_code:
                prompt += "โค้ดอ้างอิง:\n"
                for c in q_code:
                    prompt += f"```\n{c}\n```\n\n"
                    
            if q_choices:
                prompt += "ตัวเลือก:\n"
                for i, choice in enumerate(q_choices):
                    prompt += f"{i+1}. {choice}\n"
            
            contents = [prompt] + q_imgs 
            
            try:
                response = model.generate_content(contents)
                ai_answer = response.text
            except Exception as e:
                ai_answer = f"**เกิดข้อผิดพลาดในการเรียก API:** {e}"
            
            md_file.write(f"## ข้อที่ {q_idx + 1}\n\n")
            md_file.write(f"**โจทย์:** {q_text}\n\n")
            
            if q_code:
                md_file.write("**โค้ดในโจทย์:**\n")
                for c in q_code:
                    md_file.write(f"```python\n{c}\n```\n\n")
            
            if q_imgs:
                md_file.write(f"*(ข้อนี้มีรูปภาพประกอบ {len(q_imgs)} รูป ซึ่งถูกส่งให้ AI ประมวลผลแล้ว)*\n\n")
                
            if q_choices:
                md_file.write("**ตัวเลือก:**\n")
                for i, choice in enumerate(q_choices):
                    md_file.write(f"- {choice}\n")
            md_file.write("\n")
            
            md_file.write("**🤖 คำตอบจาก Gemini:**\n")
            md_file.write(f"{ai_answer}\n\n")
            md_file.write("---\n\n") # ขีดเส้นคั่นข้อ
            
            time.sleep(4) 

print("\n✅ ดำเนินการเสร็จสิ้น! ตรวจสอบไฟล์ .md ในโฟลเดอร์ได้เลยครับ")

เริ่มส่งข้อมูลให้ Gemini วิเคราะห์...

กำลังประมวลผลคอร์ส: รู้จักปัญญาประดิษฐ์ และการเรียนรู้ของเครื่อง...
  - กำลังตอบข้อที่ 1/10


KeyboardInterrupt: 